# 🛡️ SecureOps Assistant — Basic RAG Pipeline

A minimal, end-to-end RAG pipeline for industrial (OT/ICS) cybersecurity Q&A.

**Steps:**
0. Install dependencies & connect to Gemini
1. Download the corpus (NIST PDFs + CISA ICS advisories)
2. Parse & chunk the documents
3. Embed & index in ChromaDB
4. Retrieve relevant chunks
5. Generate a grounded, cited answer (refuses when unsupported)

## Step 0 — Install dependencies & connect to Gemini

In [ ]:
# Install the libraries we need (quiet).
%pip install -q chromadb sentence-transformers pypdf google-genai requests beautifulsoup4 lxml
print("✅ Dependencies installed")

In [ ]:
import os

# Load the Gemini API key: from Colab Secrets if available, else from env var, else prompt.
API_KEY = os.environ.get("GOOGLE_API_KEY")
try:
    from google.colab import userdata  # only exists inside Colab
    API_KEY = userdata.get("GOOGLE_API_KEY")
except Exception:
    pass
if not API_KEY:
    from getpass import getpass
    API_KEY = getpass("Paste your Gemini API key (input hidden): ")

# Create the Gemini client and pick a free-tier model.
from google import genai
client = genai.Client(api_key=API_KEY)
GEN_MODEL = "gemini-2.5-flash"

# Smoke test: one tiny call to confirm the key works.
r = client.models.generate_content(model=GEN_MODEL, contents="Reply with exactly: OK")
print("✅ Gemini connected:", r.text.strip())

## Step 1 — Download the corpus

Two public-domain NIST PDFs plus a sample of CISA ICS advisories (with a bundled fallback so the notebook always runs).

In [ ]:
import requests, pathlib

# Folder to hold the downloaded corpus.
CORPUS_DIR = pathlib.Path("corpus")
CORPUS_DIR.mkdir(exist_ok=True)
HEADERS = {"User-Agent": "Mozilla/5.0 (SecureOps student notebook)"}

# The two NIST PDFs.
PDFS = {
    "nist_sp800_82r3.pdf": "https://nvlpubs.nist.gov/nistpubs/SpecialPublications/NIST.SP.800-82r3.pdf",
    "nist_csf_2_0.pdf":    "https://nvlpubs.nist.gov/nistpubs/CSWP/NIST.CSWP.29.pdf",
}

# Download each PDF once (skip if already present).
for fname, url in PDFS.items():
    dest = CORPUS_DIR / fname
    if dest.exists():
        print(f"✅ already downloaded: {fname}")
        continue
    resp = requests.get(url, headers=HEADERS, timeout=120)
    resp.raise_for_status()
    dest.write_bytes(resp.content)
    print(f"✅ saved {fname} ({len(resp.content)/1e6:.1f} MB)")

In [ ]:
from bs4 import BeautifulSoup
import json, time

# Fetch a sample of CISA ICS advisories from their RSS feed.
N_ADVISORIES = 20
FEED_URL = "https://www.cisa.gov/cybersecurity-advisories/ics-advisories.xml"

advisories = []
try:
    feed = requests.get(FEED_URL, headers=HEADERS, timeout=60)
    feed.raise_for_status()
    items = BeautifulSoup(feed.content, "xml").find_all("item")[:N_ADVISORIES]
    for it in items:
        title, link = it.title.get_text(strip=True), it.link.get_text(strip=True)
        try:
            page = requests.get(link, headers=HEADERS, timeout=60)
            page.raise_for_status()
            soup = BeautifulSoup(page.content, "lxml")
            main = soup.find("main") or soup.body
            text = " ".join(main.get_text(" ", strip=True).split())
            if len(text) > 500:
                advisories.append({"title": title, "url": link, "text": text})
            time.sleep(1)  # be polite to CISA
        except Exception as e:
            print(f"  ⚠️ skipped {link}: {e}")
except Exception as e:
    print("⚠️ Could not fetch the CISA feed:", e)

# Fallback advisories so the pipeline always runs end-to-end.
if len(advisories) < 3:
    advisories = [
        {"title": "ICSA-FALLBACK-01: Example PLC Hardcoded Credentials", "url": "fallback://01",
         "text": "Example PLC family, CVSS 9.8. Hardcoded credentials (CWE-798) let a remote attacker modify control logic. Mitigations: update firmware, isolate control networks behind firewalls, use VPNs for remote access."},
        {"title": "ICSA-FALLBACK-02: Example HMI Path Traversal", "url": "fallback://02",
         "text": "Example HMI product, path traversal (CWE-22), CVSS 7.5, allows reading arbitrary files. Mitigations: upgrade, restrict network access, monitor file access, apply defense-in-depth."},
        {"title": "ICSA-FALLBACK-03: Example Historian SQL Injection", "url": "fallback://03",
         "text": "Example historian server, SQL injection (CWE-89), CVSS 8.6, in the web reporting interface. Mitigations: apply hotfix, enforce least privilege, audit logs, segment historians in a DMZ."},
    ]

# Save the advisories alongside the PDFs.
(CORPUS_DIR / "cisa_advisories.json").write_text(json.dumps(advisories, indent=2))
print(f"✅ corpus ready: 2 NIST PDFs + {len(advisories)} CISA advisories")

## Step 2 — Parse & chunk the documents

Extract text per PDF page, then cut everything into fixed-size character chunks with overlap. Each chunk keeps `source` + `page` metadata so answers can cite their origin.

In [ ]:
from pypdf import PdfReader
import json

def extract_pdf_pages(path):
    # Return one {source, page, text} dict per non-empty page.
    reader = PdfReader(str(path))
    pages = []
    for i, pg in enumerate(reader.pages, start=1):
        text = (pg.extract_text() or "").strip()
        if len(text) > 80:  # skip near-empty pages
            pages.append({"source": path.name, "page": i, "text": " ".join(text.split())})
    return pages

# Collect all document units (PDF pages + advisories).
documents = []
for fname in PDFS:
    pages = extract_pdf_pages(CORPUS_DIR / fname)
    documents.extend(pages)
    print(f"✅ {fname}: {len(pages)} pages of text")

for adv in json.loads((CORPUS_DIR / "cisa_advisories.json").read_text()):
    documents.append({"source": f"CISA: {adv['title']}", "page": 1, "text": adv["text"]})
print(f"✅ total document units: {len(documents)}")

In [ ]:
CHUNK_SIZE = 1000     # characters per chunk
CHUNK_OVERLAP = 150   # characters shared between neighbouring chunks

def naive_chunk(text, size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    # Simple fixed-size character chunking with overlap.
    chunks, start = [], 0
    while start < len(text):
        chunks.append(text[start:start + size])
        start += size - overlap
    return chunks

# Build parallel lists of chunk text and its metadata.
chunks, metadatas = [], []
for doc in documents:
    for piece in naive_chunk(doc["text"]):
        chunks.append(piece)
        metadatas.append({"source": doc["source"], "page": doc["page"]})

print(f"✅ {len(chunks)} chunks (size={CHUNK_SIZE}, overlap={CHUNK_OVERLAP})")

## Step 3 — Embed & index in ChromaDB

Embed every chunk with `all-MiniLM-L6-v2` and store the vectors + metadata in a persistent ChromaDB collection. This is the slow cell; it only needs to run once per runtime.

In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer

PERSIST_DIR = "./chroma_db"
EMBED_MODEL = "all-MiniLM-L6-v2"

# Load the embedding model and open a persistent Chroma collection.
embedder = SentenceTransformer(EMBED_MODEL)
chroma = chromadb.PersistentClient(path=PERSIST_DIR)
collection = chroma.get_or_create_collection("secureops", metadata={"hnsw:space": "cosine"})

# Embed and add chunks in batches (skip if the index is already built).
if collection.count() >= len(chunks):
    print(f"✅ index already built ({collection.count()} chunks)")
else:
    BATCH = 256
    for i in range(0, len(chunks), BATCH):
        batch_docs = chunks[i:i+BATCH]
        embs = embedder.encode(batch_docs, show_progress_bar=False).tolist()
        collection.add(
            ids=[f"chunk-{j}" for j in range(i, i + len(batch_docs))],
            documents=batch_docs,
            embeddings=embs,
            metadatas=metadatas[i:i+BATCH],
        )
        print(f"  indexed {min(i+BATCH, len(chunks))}/{len(chunks)}", end="\r")
    print(f"\n✅ index built: {collection.count()} chunks")

## Step 4 — Retrieve

Embed the question and return the top-*k* most similar chunks.

In [ ]:
TOP_K = 5

def retrieve(question, k=TOP_K):
    # Return the k most similar chunks as (text, metadata, distance).
    q_emb = embedder.encode([question]).tolist()
    res = collection.query(query_embeddings=q_emb, n_results=k)
    return list(zip(res["documents"][0], res["metadatas"][0], res["distances"][0]))

# Quick look at what retrieval returns.
for text, meta, dist in retrieve("What does NIST recommend regarding remote access to OT networks?"):
    print(f"[{meta['source']} p.{meta['page']}] (distance {dist:.3f})")
    print("   ", text[:160].replace("\n", " "), "...\n")

## Step 5 — Generate a grounded, cited answer

Pass the retrieved chunks to Gemini as numbered context blocks. The prompt enforces: cite every claim, and refuse when the context doesn't contain the answer.

In [ ]:
import time

# System prompt: grounding + citation + refusal rules.
SYSTEM_PROMPT = """You are SecureOps Assistant, helping a junior security analyst understand industrial (OT/ICS) cybersecurity.

Rules:
1. Answer ONLY using the numbered context blocks. Do not use outside knowledge.
2. Cite the supporting block number(s) after each claim, like [1] or [2][3].
3. If the context does not contain the answer, reply exactly:
   "I don't have enough information in my knowledge base to answer that."
4. Be concise. Never invent products, numbers, or recommendations."""

def build_prompt(question, hits):
    # Format retrieved chunks as numbered, source-labelled context blocks.
    blocks = "\n\n".join(
        f"[{i}] (source: {meta['source']}, page {meta['page']})\n{text}"
        for i, (text, meta, _) in enumerate(hits, start=1)
    )
    return f"{SYSTEM_PROMPT}\n\n=== CONTEXT BLOCKS ===\n{blocks}\n\n=== QUESTION ===\n{question}\n\n=== ANSWER ==="

def ask_secureops(question, k=TOP_K, retries=3):
    # Retrieve, build the prompt, then generate (with a simple retry for rate limits).
    hits = retrieve(question, k)
    prompt = build_prompt(question, hits)
    for attempt in range(retries):
        try:
            resp = client.models.generate_content(model=GEN_MODEL, contents=prompt)
            answer = resp.text.strip()
            break
        except Exception as e:
            wait = 20 * (attempt + 1)
            print(f"  ⚠️ API error ({e}); retrying in {wait}s ...")
            time.sleep(wait)
    else:
        return "ERROR: generation failed after retries."

    # Append the list of sources that were retrieved.
    sources = "\n".join(
        f"  [{i}] {meta['source']} (p.{meta['page']})"
        for i, (_, meta, _) in enumerate(hits, start=1)
    )
    return f"{answer}\n\n--- Sources retrieved ---\n{sources}"

print(ask_secureops("What does NIST recommend regarding remote access to OT networks?"))